# DQN for 2048

This notebook contains an implementation of a deep Q-network (DQN) to learn to play the game 2048. As the name suggests, DQNs use deep neural networks to approximate the Q-function, a central concept in reinforcement learning. 

## The Theory

Before we get into the code, let us formulate the problem, understand the solution strategy, and how exactly deep learning is used.

### What is Reinforcement Learning?

Reinforcement learning is used in scenarios where an agent has to learn its environment and respond to it in some optimal fashion. For instance, in 2048, the agent has to learn about the game, its rules, and objectives before determining the optimal set of moves that will help achieve these objectives. Unlike in traditional optimization problems where the end objective is known, we _do not_ tell the agent about the end goals or even the rules of the game (beyond what it can implicitly learn by interacting with the game engine).

The agent has to follow a policy of "explore and exploit". It should make exploratory moves of uncertain rewards to understand the overall landscape while also making the moves that it thinks are optimal (most rewards). RL is all about doing these efficiently and optimally - make too many exploratory moves and you are forever wandering the landscape without ever getting better; be too aggressive with making "optimal" moves without learning enough of the landscape and the agent will stay hopelessly myopic, unable to exploit the long game.

### Briefly about 2048

The game itself is fairly intuitive but let us highlight the rules of the game which are relevant for our implementation.
* A game starts with a blank board with two random tiles (2 tile with $90\%$ probability and 4 tile with $10\%$ probability) at random locations. After a move, all tiles move in that direction and occupy any free squares along the way. Should two identical tiles collide, they fuse to form a larger tile
* After every move, a new tile will spawn at random in one of the empty squares. This will be a 2 tile with 90% probability and a 4 tile with 10% probability (Aside: I have lost so many games when a 4 tile unexpectedly pops up and ends the game!!)
* The stated objective of the game is to reach 2048 but any move that merges two tiles gets a reward equal to the value of the merged tiles.

### Imports and display helper functions

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import random
import time
import os
import json
import csv
from collections import Counter
from pathlib import Path
import math
from IPython.display import HTML, display

In [2]:
_TILE_COLORS = {
    0:    ('#cdc1b4', '#cdc1b4'),   # empty
    2:    ('#eee4da', '#776e65'),
    4:    ('#ede0c8', '#776e65'),
    8:    ('#f2b179', '#f9f6f2'),
    16:   ('#f59563', '#f9f6f2'),
    32:   ('#f67c5f', '#f9f6f2'),
    64:   ('#f65e3b', '#f9f6f2'),
    128:  ('#edcf72', '#f9f6f2'),
    256:  ('#edcc61', '#f9f6f2'),
    512:  ('#edc850', '#f9f6f2'),
    1024: ('#edc53f', '#f9f6f2'),
    2048: ('#edc22e', '#f9f6f2'),
}
_BIG = ('#3c3a32', '#f9f6f2')


def show_board(board, title=None, size=68, highlight=None):
    """
    Render a 4x4 2048 board as HTML in a notebook.

    board     : (4,4) array-like of ints
    title     : optional caption above the grid
    size      : cell size in px
    highlight : optional flat index (0-15) or (row, col) to outline
    """
    b = np.asarray(board)
    if highlight is not None and not isinstance(highlight, tuple):
        highlight = (highlight // 4, highlight % 4)

    cells = []
    for r in range(4):
        for c in range(4):
            v = int(b[r, c])
            bg, fg = _TILE_COLORS.get(v, _BIG)
            # shrink font for wide numbers so 1024/2048 still fit
            fs = size * (0.42 if v < 100 else 0.32 if v < 1000 else 0.26)
            ring = ('box-shadow:inset 0 0 0 3px #e8412f;'
                    if highlight == (r, c) else '')
            cells.append(
                f'<div style="width:{size}px;height:{size}px;background:{bg};'
                f'color:{fg};border-radius:{size*0.09:.0f}px;display:flex;'
                f'align-items:center;justify-content:center;font-weight:700;'
                f'font-size:{fs:.0f}px;font-family:system-ui,sans-serif;'
                f'{ring}">{v if v else ""}</div>'
            )

    cap = (f'<div style="font:600 13px system-ui,sans-serif;color:#444;'
           f'margin-bottom:6px">{title}</div>') if title else ''

    display(HTML(
        f'<div style="display:inline-block">{cap}'
        f'<div style="display:grid;grid-template-columns:repeat(4,{size}px);'
        f'gap:{size*0.11:.0f}px;background:#bbada0;padding:{size*0.11:.0f}px;'
        f'border-radius:{size*0.13:.0f}px;width:max-content">'
        f'{"".join(cells)}</div></div>'
    ))

def show_boards(boards, titles=None, size=58, gap=22):
    """Render several boards in a row. `boards` is a list of (4,4) arrays."""
    titles = titles or [None] * len(boards)
    blocks = []
    for b, t in zip(boards, titles):
        b = np.asarray(b)
        cells = []
        for r in range(4):
            for c in range(4):
                v = int(b[r, c])
                bg, fg = _TILE_COLORS.get(v, _BIG)
                fs = size * (0.42 if v < 100 else 0.32 if v < 1000 else 0.26)
                cells.append(
                    f'<div style="width:{size}px;height:{size}px;background:{bg};'
                    f'color:{fg};border-radius:{size*0.09:.0f}px;display:flex;'
                    f'align-items:center;justify-content:center;font-weight:700;'
                    f'font-size:{fs:.0f}px;font-family:system-ui,sans-serif;">'
                    f'{v if v else ""}</div>'
                )
        cap = (f'<div style="font:600 12px system-ui,sans-serif;color:#444;'
               f'margin-bottom:5px;text-align:center">{t}</div>') if t else ''
        blocks.append(
            f'<div>{cap}<div style="display:grid;'
            f'grid-template-columns:repeat(4,{size}px);gap:{size*0.11:.0f}px;'
            f'background:#bbada0;padding:{size*0.11:.0f}px;'
            f'border-radius:{size*0.13:.0f}px">{"".join(cells)}</div></div>'
        )
    display(HTML(
        f'<div style="display:flex;gap:{gap}px;align-items:flex-start">'
        f'{"".join(blocks)}</div>'
    ))

### RL and 2048

RL is a natural choice for teaching an agent to solve 2048 but it is also a brutal environment for 2048. To see why, it helps to understand exactly what the agent has access to.

The RL agent has access to a game engine which it can query at all times. The game engine takes the current state of the board and the agent's action (UP, DOWN, LEFT, or RIGHT) and returns two outputs:
* The reward, if any, for the agent's action
* The new state of the board after all the tiles have moved and merged _and_ a new tile has spawned at random

In [3]:
sample_board = np.array([[4,512,2,2], [32,128,16,0], [8,16,2,0], [2,0,0,2]], dtype=np.int32)
next_board = np.array([[0,4,512,4], [0,32,128,16], [0,8,16,2], [0,2,0,4]], dtype=np.int32)
show_boards([sample_board, next_board])

In this example, the agent takes the board on the left and performs a RIGHT move. The game engine would return a reward of $8$ (the tiles in the top and bottom rows merged) and the board on the right (with a new 2 tile spawned at the location $(4,2)$)

Note what the agent **does not** have access to:
* It does not know the dynamics of how the tiles move. It is not told that the 512 tile will move from $(1,2)$ to $(1,3)$ if it moves RIGHT
* It does not know that collision of identical tiles leads to a merge
* It does not know that two colliding 2 tiles will always produce a 4 tile or two colliding 32 tiles will produce a 64 tile. It is blind to this math
* It does not know the dynamics of a new tile spawn or the probability that it will be a 2 tile or 4 tile

The _only_ way for the agent to learn these dynamics is through the reward. As if this weren't hard enough, the unique dynamics of 2048 introduces many other complexities that make it a challenging problem for RL.
* **Exponentially sparse rewards**: Since the agent is not driven by any global objective or game knowledge, till such time as the agent combines two 64 tiles to form a 128 tile, the agent is _not even aware_ that a reward of $128$ is possible, much less that it is worth working towards. But to get to a 128 tile, the agent needs to form two 64 tiles which will take at least twice as long as forming one 64 tile (probably longer since forming the second 64 tile is harder than forming the first one). Building this out, we see that it will take 4 times as long to get a reward of 256 and 8 times as long to get a reward of 512 - the agent should somehow survive long enough through these phases _without_ knowing that such persistence might get rewarded. 
* **Stochastic transitions**: After every move, a random new tile spawns which means the game is non-deterministic. This means that we cannot make unambiguous statements about what a "good" move is or how to quantify it - the best we can hope for is a statistical averaging over all possible future random tile spawns. This means we need more samples per state to get a robust estimate of the optimal move for that state.
* **Combinatorial state space**: To some extent, this is common to all non-trivial games but in 2048, every state is almost guaranteed to be a previously unseen one simply because as the game goes on, larger and larger tiles form (which were never seen before). This means that any strategy has to be driven by relative features (for ex., are two adjacent tiles identical?) rather than absolute (are two adjacent tiles both 32-tiles?)

### Formalizing the game

We can capture the board as a 16-dimensional vector of numbers from $\{0, 2,4,8,\dots,1024,2048\}$ where the 0 tile means an empty square. Call this state as $s$. At any stage, the agent has $4$ actions available to it (some of which might be invalid in the sense that moves in that direction do not change the board at all). Call this action $a$. In the language of $Q$-functions, we are trying to learn a function $Q(s,a)$ that will take these two arguments as input and returns the total reward that we can get by performing the action $a$ on state $s$. Note that this is not just the immediate reward (which the game engine will tell us) but all the rewards summed up from that point onwards till the end of the game. Two points to note here:
* Like we have seen before, the game is stochastic because of tile spawning. So, we should think of $Q(s,a)$ as the _expected_ future reward
* For technical reasons, future rewards need to be discounted. This is done for a variety of reasons - numerical stability, limiting the horizon over which the agent has to reason etc. It can also be thought of as a bias-variance tradeoff parameter - the closer the discount factor is to 1, the lower the bias (the reward is as close to the true reward as possible) but higher the variance (the model has to estimate ultra-noisy rewards several tens of moves away). Agents trainied with small discount factors will learn fast but will tend to stay myopic.

Assuming that the game ends at the 2048 tile (it doesn't have to), each square can take one of $11$ values (empty or $2^k$ for $k=1,\dots,10$) which gives us an upper bound of $11^{16} \approx 4.6 \times 10^{16}$ possible states. Not all of these will be reachable in a valid game but this is a reasonable bound. Therefore, $Q(s,a)$ is just an astronomically large table with $11^{16}$ rows and $4$ columns (one for each action). If we can somehow populate this table with values, we have solved the game. At every stage, we consult the Q-table for the state corresponding to the board at that stage, read-off the expected rewards for each of the $4$ actions, and choose the action with the highest reward. The game then moves (stochastically) forward and we repeat the process with the new board and so on.

### Reward functions and the Bellman equation

Mathematically, the $Q$ function is defined as $$Q_t(s,a) = \mathbb{E}\left[\sum_{k=0}^{\infty} \gamma^k R_{t+k+1} \Bigm| s_t = s, a_t = a \right]$$ where the expectation is over future random tile spawns. It should be intuitive to see that:
* The optimal policy does not depend on $t$. Only the state of the board matters, not how many moves it took to get there. Note that this is a property of 2048 that won't always extend to other games. For example, in games where the user is allowed only a finite number of moves, this stationarity assumption clearly won't hold.
* The optimal policy can be described by the recursive equation $$Q^{*}(s,a) = \mathbb{E} \left[ r + \gamma \max_{a'} Q^{*}(s',a')\Bigm| s_t = s, a_t = a \right]$$ where $r$ is the immediate reward from the game engine and the other term is the best possible reward achievable from the new game state. This is the *Bellman equation* from dynamic programming and Markov decision processes (MDP).

Let us tie this back to the discussion earlier about $Q(s,a)$. We saw that this is a giant table of dimension $11^{16} \times 4$ and the equation above gives us a system of equations that connects each cell of this table with other cells of this table. If we can "solve" this system of equations, we are done but therein lies the rub. Even setting aside the sheer number of variables involved, there are two operators to contend with - the expectation and the maximum. The former can be expanded out as a linear combination of $Q(s',a')$ but there is no way around the maximization which makes this system highly non-linear.

So, how do we compute $Q(s,a)$? Or equivalently, given a board $s$, how do we compute the discounted future rewards $Q(s,\cdot)$ associated with all $4$ actions that can be performed on that board? Our approach is to train a deep network to "learn" $Q(s)$ - such networks are good high dimensional learners after all. But what do we train this network on?

### From Bellman to CNNs

Imagine for a moment that a genie is able to evaluate the future discounted reward for any board and any action. 

<video autoplay muted playsinline controls preload="auto" width="760">
  <source src="figures/S1QDefinition.mp4" type="video/mp4">
</video>

*Figure 1: Predicting future discounted reward for action $a$ on state $s$.*

We can then train a neural network to mimic this genie. Use the genie to query several boards and actions $(s,a)$, get the resulting reward $Q(s,a)$, and train a network (called the policy net) to take $(s,a)$ as input and predict $Q(s,a)$ as output. The exact architecture of this network is not important here, just the concept is.

<video autoplay muted playsinline controls preload="auto" width="760">
  <source src="figures/S2PolicyTraining.mp4" type="video/mp4">
</video>

*Figure 2: Train the policy net to predict $Q(s,a)$*

But we know that this infinite expectation is equivalent to the recursive Bellman equation.

<video autoplay muted playsinline controls preload="auto" width="760">
  <source src="figures/S3BellmanSplit.mp4" type="video/mp4">
</video>

*Figure 3: Telescoping form of $Q(s,a)$*

You might wonder if we aren't simply deluding ourselves - if good training labels exist for the policy net to train on, there is no need for the policy net in the first place. But let us keep up this fiction for a bit longer. Note that one of the two terms in this reward function is easy to get. It is just the immediate reward for the action $a$ on the board $s$ which the game engine can provide. 

<video autoplay muted playsinline controls preload="auto" width="760">
  <source src="figures/S4SplitTerms.mp4" type="video/mp4">
</video>

*Figure 4: The two components of $Q(s,a)$*

The big question is the second term - we need a good estimate of $Q(\cdot, \cdot)$ to evaluate the second term so that we can add it to the immediate reward to get a good label for the policy net to train on so that we get a good estimate of $Q(\cdot, \cdot)$. How do we escape this infinite loop?

Here is the big idea: **We will use a "target net" to evaluate $Q(s',a')$**. 

<video autoplay muted playsinline controls preload="auto" width="760">
  <source src="figures/S5TargetNet.mp4" type="video/mp4">
</video>

*Figure 5: Use a target net to predict $Q(s',a')$*

There are a couple of important points about this target net:
* It does not learn by gradient descent. Instead, it is a "low-pass filter" version of the policy net. Mathematically, we do $$\theta_{\text{target}} \leftarrow \tau \cdot \theta_{\text{policy}} + (1-\tau) \theta_{\text{target}}$$ for a very small value of $\tau$ ($0.005$ in the code below)
* This is an highly smoothened version of the policy net. A good way to think about it is that the target net will only learn what the policy net is sure of. In a stochastic environment like 2048, this translates to the target net only learning those rewards that it is very sure of. We will see examples of what this might look like in the next section.

<video autoplay muted playsinline controls preload="auto" width="760">
  <source src="figures/S6SoftUpdate.mp4" type="video/mp4">
</video>

*Figure 6: Transfer learnings from policy to target net*

Put everything together and this is what the architecutre looks like

<img src="figures/StaticDiagram.png" style="width:700px; max-width:100%">

### Architecture explained with an example

Here is an example to illustrate how the model works. We make several simplifying assumptions for the sake of illustration here. The code itself assumes none of this.

* All new tiles that spawn will be 2 tiles
* We will walk through the training as though it happens in distinct stages with each stage causally following from the previous ones
* We will pretend that we know exactly what the network learns and when.

In [4]:
final_board = np.array([[2,16,4,8], [4,2,32,2], [8,16,2,16], [2,2,8,4]], dtype=np.int32)
penultimate_board = np.array([[2,16,4,8], [4,2,32,2], [8,16,2,16], [2,8,2,2]], dtype=np.int32)
pre_penultimate_board = np.array([[2,16,2,8], [4,2,2,2], [8,16,32,16], [2,8,2,2]], dtype=np.int32)

show_board(final_board)

#### Stage 1 of the training

This board has been selected specifically because it allows only $2$ valid moves - a LEFT or a RIGHT each with a reward of $4$ and both of which will end the game (assuming that the newly spawned tile is a 2 tile). This means that it is possible to compute $Q(s,a)$ accurately for this state $s$ since there is no discounted future reward to worry about (the part that the target net is supposed to calculate). That means that this is a clear signal that the policy net can learn. This is shown graphically below.

<video autoplay muted playsinline controls preload="auto" width="760">
  <source src="figures/E1NoBootstrap.mp4" type="video/mp4">
</video>

*Figure 1: Move LEFT and the game ends*

**Aside**: In actuality, if you play LEFT and a 4 tile spawns in $(4,4)$, a fully trained agent can easily progress all the way to 2048 tile which goes to show how complicated the game dynamics is. $Q(s,a)$ is hard to compute even for such a completely filled, tightly constrained board.

In the course of its training, the policy network sees many states that look like this (for ex., maybe a different state $s$ has the 8 tile in the $(1,1)$ square) and is able to calculate the exact reward for each of them. As the network sees enough such examples, it learns (through CNN kernels) that when tiles are jammed up in the first $3$ rows and the $4^{\text{th}}$ row looks like this, it should predict a $4$. Of course, this is just an intuitive crutch - the actual mechanics are hidden inside the CNN black box.

In this way, the policy network slowly learns to be sure of the end-states where it knows the reward for certain and learns to predict them. But the policy net treats all states equally - it has no way of knowing which is an end-state and which isn't. It tries equally hard to predict the label for all other states as well but crucially, the rewards for the other states will not have a consistent pattern. The policy net might fluctuate widely as it tries to minimize the loss in each minibatch but with boards like this, a consistent pattern emerges.

**Here is the crucial bit of insight**: When the policy net is low-pass filtered and transferred to the target net, all the fluctuations go away but the consistent end-game signals remain. And that means the target net can now reliably score boards that are one move away from Game Over.

#### Stage 2 of the training

We can now play out the Bellman recursion backwards. Consider the following grid:

In [5]:
show_board(penultimate_board)

If we perform a RIGHT move on this board, we will get the final board above. Once the target net knows how to score the final board consistently, the policy net starts getting a reliable label for this board. Once again, this signal will be consistent amidst all the noisy labels and the policy net will slowly adjust its weights till it reliably matches the computed reward for this board.

This process is illustrated in the animation below.

<video autoplay muted playsinline controls preload="auto" width="760">
  <source src="figures/E2BootstrapOnce.mp4" type="video/mp4">
</video>

*Figure 2: Move RIGHT to reach the previous board*

#### Stage 3 of the training

We can keep playing this movie backwards and see this process play out
* The target net gets good at scoring some states, say $\mathcal{A}$
* This provides the policy net good labels for states that are upstream of $\mathcal{A}$. Call them $\mathcal{A}_1$
* The policy net gets good at predicting $Q(s,a)$ for $\mathcal{A}_1$
* The target net inherits this knowledge from the policy net and gets good at scoring the states $\mathcal{A}_1$
* The target net gets good at scoring the states, $\mathcal{A} \cup \mathcal{A}_1$
* And the process repeats all over again...

Here is one more example where an UP move from this board gets you to the board in stage 2

<video autoplay muted playsinline controls preload="auto" width="760">
  <source src="figures/E3BootstrapTwice.mp4" type="video/mp4">
</video>

*Figure 3: Move UP to reach the previous board*

## The Code

Let us translate this theory into working code. The broad ideas and code-specific modifications are listed below:
* Training the policy net after every move is highly inefficient. Instead, play a bunch of moves, fill up a memory buffer with both the moves and the estimated reward, choose a mini-batch from this buffer, and train the policy net on this batch.
* To estimate a reward at any stage, we combine the immediate reward from the game engine with future rewards as estimated by the target net
* So far, we have kept the model fairly generic. In this implementation, we will use a **CNN** (which treats the board as a $4 \times 4 \times 1$ image and uses a $2 \times 2$ kernel on it) but this is not a necessary condition. Any network with sufficient learning capacity will do the job.
* **Exploration Strategy**: We haven't yet discussed the "exploration" part of RL. In the beginning, the networks are just a bunch of random weights and rather than trust their every action recommendation, we are better off exploring the search space. This exploration helps the agent learn the parameters of the problem (remember that the agent does not know anything about the rules of the game) and also potentially reaches the kind of conclusive end-states (like in the examples above) from where definite learning can begin. The exploration strategy is quite simple - at each stage, with probability $\epsilon$, we will make a random move and with probability $(1-\epsilon)$, we will follow the agent's recommendation. When we start, $\epsilon$ will be close to $1$ and as the network learns, we will decay this $\epsilon$ to close to $0$. There are more sophisticated strategies possible (like overlaying a cosine jitter on top of this exponential decay), but we will keep things simple here.

### 1. Hyperparameters

In [6]:
BATCH_SIZE    = 64   # For each SGD update, use this many random moves from the memory buffer
GAMMA         = 0.99 # discount rate for future rewards which gives the network a window of around 460 moves (0.99^460 ~ 1% ) or one typical game
EPS_START     = 1.0
EPS_END       = 0.01
EPS_DECAY     = 2000000 # Window over which epsilon will decay from its start value to end value
LEARNING_RATE = 1e-4 # SGD learning rate

# A typical game is around 500 moves and so this buffer will hold 200 games worth of moves
MEMORY_SIZE = 100_000

# For every 4 moves we put into the buffer, we train on a random batch of samples. 
# Note that the two are not directly related - the 4 moves are from the current game being played 
# while the BATCH_SIZE is drawn at random from the memory buffer and will be a random subset from past games
# That doesn't mean we can completely decouple filling the buffer and training on the buffer
# because we don't want the model that generated the buffer moves to drift away from the latest training weights
# A 1:1 ratio might be an overkill but 1:4 works well. Might be able to push it to 1:8 even
OPTIMIZE_EVERY = 4

### 2. Replay Memory

Define the class that holds the buffer of past moves for the model to train on

In [7]:
class TensorReplayBuffer:
    def __init__(self, capacity, device):
        self.capacity = capacity    # Store these many boards (not games) in memory
        self.device   = device
        self.ptr      = 0      
        self.size     = 0

        # Pre-allocate on CPU; batches are moved to device at sample time.
        self.states      = torch.zeros(capacity, 1, 4, 4) # Batch x Channels x Height x Width that Conv2d expects even though "channel" is trivially 1 here
        self.actions     = torch.zeros(capacity, 1, dtype=torch.long) # torch.gather() later needs a 2D input
        self.next_states = torch.zeros(capacity, 1, 4, 4)
        self.rewards     = torch.zeros(capacity)
        self.dones       = torch.zeros(capacity)
        self.next_masks  = torch.zeros(capacity, 4, dtype=torch.bool)

    # Push a board into memory 
    # Contains present board, chosen action, resultant next board, reward for this action, whether this move ends the game, and valid moves for next board
    # next_mask is stored here so optimize_model() can mask invalid next-state actions when computing Bellman targets
    def push(self, state, action, next_state, reward, done, next_mask):
        i = self.ptr
        self.states[i]      = state.squeeze(0)
        self.actions[i]     = action
        self.next_states[i] = next_state.squeeze(0)
        self.rewards[i]     = reward
        self.dones[i]       = done
        self.next_masks[i]  = torch.from_numpy(next_mask)
        self.ptr  = (i + 1) % self.capacity     # Circular write pointer
        self.size = min(self.size + 1, self.capacity) # How much of the buffer is full

    # Pick a set of random boards from memory (equal to batch size) and send all their parameters
    def sample(self, batch_size):
        """Return a dict of pre-batched tensors on the training device."""
        idx = torch.randint(0, self.size, (batch_size,))
        return dict(
            states      = self.states[idx].to(self.device),
            actions     = self.actions[idx].to(self.device),
            next_states = self.next_states[idx].to(self.device),
            rewards     = self.rewards[idx].to(self.device),
            dones       = self.dones[idx].to(self.device),
            next_masks  = self.next_masks[idx].to(self.device),
        )

    def __len__(self):
        return self.size


### 3. Game Engine

This is the class that implements the game dynamics and calculates immediate rewards. There are some interesting ideas here about rotating the board to efficiently manage the moves.

In [11]:
class Game2048:
    def __init__(self):
        self.size = 4
        self.reset()

    # Create a new board. A new board begins with two randomly populated tiles
    def reset(self):
        self.board = np.zeros((self.size, self.size), dtype=int)
        self.add_new_tile()
        self.add_new_tile()
        return self.board.copy()

    # A new tile is populated uniformly in one of the free positions at random
    # 90% chance of being a 2 tile and 10% chance of being a 4 tile
    def add_new_tile(self):
        empty_cells = list(zip(*np.where(self.board == 0)))
        if empty_cells:
            r, c = random.choice(empty_cells)
            self.board[r, c] = 2 if random.random() < 0.9 else 4
        return (4*r + c)

    def get_state(self):
        return self.board.copy()

    # Only used internally to avoid dealing with any move that will leave the board unchanged
    # For ex., if the only populated tiles are (0,0) and (1,0), LEFT is useless
    @staticmethod
    def _can_move(board, action):
        """Return True if the given action would change the board."""
        temp_board = np.rot90(board, action)
        for r in range(4):
            for c in range(1, 4):
                if temp_board[r, c] != 0:
                    if temp_board[r, c - 1] == 0:
                        return True
                    if temp_board[r, c - 1] == temp_board[r, c]:
                        return True
        return False

    def get_valid_moves(self):
        """Return a boolean mask (4,) — True where an action is legal."""
        return np.array([self._can_move(self.board, a) for a in range(4)])

    def is_game_over(self):
        return not self.get_valid_moves().any()

    # See _move() for an explanation
    def _stack(self):
        new_board = np.zeros((self.size, self.size), dtype=int)
        for i in range(self.size):
            fill_ptr = 0
            for j in range(self.size):
                if self.board[i, j] != 0:
                    new_board[i, fill_ptr] = self.board[i, j]
                    fill_ptr += 1
        self.board = new_board

    # See _move() for an explanation
    def _combine(self):
        score = 0
        for i in range(self.size):
            for j in range(self.size - 1):
                if self.board[i, j] != 0 and self.board[i, j] == self.board[i, j + 1]:
                    self.board[i, j] *= 2
                    score += self.board[i, j]
                    self.board[i, j + 1] = 0
        return score

    # Some clever hacks to make board management easy
    # Two ideas - rotation and stacking
    # 1. Rotation: Rather than write different logic for each move, use the fact that every move is a LEFT move after suitably rotating the board
    # So, rotate the board, make the LEFT move, undo the rotation by rotating in the opposite direction
    # 2. Stacking: Under LEFT, two things can happen. Tiles move leftward past any open squares till they can't move any more. 
    # If they hit an identical tile, they can combine. Stacking allows us to implement this logic while worrying about _combine for only adjacent cells
    # _stack() moves all tiles to the left. _combine() combines adjacent tiles. _stack() again fills out any remaining gaps
    # Ex.: [2, 4, 0, 4] -> _stack() -> [2, 4, 4, 0] -> _combine() -> [2, 8, 0, 0] -> _stack() -> [2, 8, 0, 0]
    # [2, 0, 4, 4] -> _stack() -> [2, 4, 4, 0] -> _combine() -> [2, 8, 0, 0] -> _stack() -> [2, 8, 0, 0]
    def move(self, direction):
        """
        Execute action.

        Returns: (next_state, reward, done, moved, next_mask)
        """
        original_board = self.board.copy()
        self.board = np.rot90(self.board, direction)
        self._stack()
        reward = self._combine()
        self._stack()
        self.board = np.rot90(self.board, -direction)
        moved = not np.array_equal(original_board, self.board)
        new_tile_loc = -1
        if moved:
            new_tile_loc = self.add_new_tile()

        # Compute next_mask once; derive done from it.
        next_mask = self.get_valid_moves()
        done = not next_mask.any()
        return self.board.copy(), reward, done, moved, next_mask, new_tile_loc


### 4. Model & Preprocessing

Define the CNN model used for the policy and target networks

In [9]:
# Apply a 2x2 kernel with stride 1 on the 4x4 image (with 1 channel) which will result in a 3x3 output
# Learn 64 such representations. Some useful representations could be when adjacent tiles are identical but we leave it to the network to figure it out
# Take the 64 channels of 3x3 images and convert them to 128 channels of 2 x 2 images (by applying a 2x2 kernel with stride of 1)
# Convert these 128 2x2 "images" into a linear vector of length 128x2x2 and pass it through a fully connected network with output 256
# Convert this 256 embedding vector into 4 numbers - one Q value for each of the 4 actions
class DQN_CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 64,  kernel_size=2, stride=1) 
        self.conv2 = nn.Conv2d(64, 128, kernel_size=2, stride=1)
        self.fc1   = nn.Linear(128 * 2 * 2, 256)
        self.fc2   = nn.Linear(256, 4)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = x.view(x.size(0), -1) # Flatten out the tensor
        x = F.relu(self.fc1(x))
        return self.fc2(x) # No relu on this layer because Q-values can be arbitrary real numbers


# Convert a board to a 1 (batch) x 1 (channel) x 4 (width) x 4 (height)
# Normalize the board so all values up to 2048 are kept within [0,1]
def preprocess_cnn(board):
    """Convert 4x4 int board → normalised [1, 1, 4, 4] float tensor."""
    log2_board = np.where(board > 0, np.log2(board), 0.0) / 11.0
    return torch.FloatTensor(log2_board).unsqueeze(0).unsqueeze(0)


### 5. Trainer

This module all the machinery to play the game as per existing policy, fill up the memory buffer, update the policy net weights, and transfer weights to target net. Beyond the deep learning components, the most interesting piece here is reward shaping where we add more terms to the base reward (which comes from the game engine) to incentivize strategies that we know to be good. This is somewhat against the more interesting goal of seeing if the agent can uncover such strategies by itself or even come up with hitherto unthought of strategies. 

In this implementation, weight updates happen on the basis of moves that might have been made several episodes earlier by a policy that is now stale (because of other intervening weight updates). This makes DQN an **off-policy** algorithm where the training data is not generated by the active policy net. This is in contrast to **on-policy** algorithms like PPO or GRPO.

In [10]:
class Trainer:
    def __init__(
        self,
        min_log_tile        = 1024,
        probs_for_logging   = None,
        resume_path         = None,
        run_folder          = None,
        stats_log_file_name = None,
        moves_log_file_name = None,
        chkpoint_suffix     = None,
        verbose             = False,
    ):
        self.device  = torch.device("cuda" if torch.cuda.is_available() else "cpu") # Find out if we have a NVIDIA GPU
        self.verbose = verbose
        self.max_tile_history = []
        self.distribution     = Counter()
        self.tau              = 0.005 # Controls how filtered the transfer from policy to target net is

        self.policy_net = DQN_CNN().to(self.device)
        self.target_net = DQN_CNN().to(self.device)
        self.target_net.load_state_dict(self.policy_net.state_dict())
        self.target_net.eval()

        self.optimizer = optim.Adam(self.policy_net.parameters(), lr=LEARNING_RATE)
        self.memory    = TensorReplayBuffer(MEMORY_SIZE, self.device)
        self.steps_this_run = 0

        self.start_episode = 0
        self.steps_done    = 0
        # ── Ability to load a previously saved checkpoint ──────────────────────────────────────────────────────────
        if resume_path and os.path.isfile(resume_path):
            print(f"=> Loading checkpoint '{resume_path}'")
            ckpt = torch.load(resume_path, map_location=self.device, weights_only=False)
            self.policy_net.load_state_dict(ckpt['state_dict'])
            self.target_net.load_state_dict(ckpt['state_dict'])
            self.optimizer.load_state_dict(ckpt['optimizer'])
            self.start_episode = ckpt.get('episode', 0)
            self.steps_done    = ckpt.get('steps_done', 0)
            print(f"=> Loaded (Step {self.steps_done})")

        self.run_folder = Path(run_folder)
        self.stats_file = self.run_folder / stats_log_file_name
        if not self.stats_file.exists():
            with open(self.stats_file, 'w', newline='') as f:
                csv.writer(f).writerow(
                    ["episode", "score", "shaped_score", "max_tile",
                     "avg_loss", "avg_q", "steps", "final_board"]
                )

        self.min_log_tile    = min_log_tile
        self.probs_for_logging = probs_for_logging
        self.move_log_path   = self.run_folder / moves_log_file_name
        self.chkpoint_suffix = chkpoint_suffix
        self.log_game_flag   = False,

    # ── Checkpointing ──────────────────────────────────────────────────────────
    def save_checkpoint(self, episode, max_tile, is_best=False):
        ckpt = {
            'episode':    episode,
            'state_dict': self.policy_net.state_dict(),
            'optimizer':  self.optimizer.state_dict(),
            'steps_done': self.steps_done,
            'max_tile':   max_tile,
        }
        s      = self.chkpoint_suffix
        latest = self.run_folder / (f"latest_model_{s}.pth" if s else "latest_model.pth")
        best   = self.run_folder / (f"best_model_{s}.pth"   if s else "best_model.pth")
        periodic_chk = self.run_folder / (f"periodic_model_{s}_ep{episode}.pth"   if s else f"periodic_model_ep{episode}.pth")
        torch.save(ckpt, latest)
        if is_best:
            torch.save(ckpt, best)
            print(f"*** New Record! Saved {best.name} (Tile {max_tile}) ***")
        if episode % 5000 == 4999:
            torch.save(ckpt, periodic_chk)
            print(f"*** Periodic saving of checkpoint! Saved {periodic_chk.name} ***")

    # ── Logging ────────────────────────────────────────────────────────────────
    def log_episode_stats(self, ep, score, shaped_score, tile, loss, q, steps, final_board=None):
        """Append episode stats to the CSV. `final_board` is serialized as JSON.

        `final_board` may be a numpy array; we convert to a nested list before JSON.
        """
        fb_serialisable = None
        if final_board is not None:
            try:
                fb_serialisable = final_board.tolist() if hasattr(final_board, 'tolist') else final_board
            except Exception:
                fb_serialisable = final_board

        with open(self.stats_file, 'a', newline='') as f:
            row = [ep, score, shaped_score, tile, f"{loss:.6f}", f"{q:.2f}", steps]
            # Append JSON string of final_board (or empty string if None)
            row.append(json.dumps(fb_serialisable, default=lambda o: o.item() if hasattr(o, 'item') else str(o)))
            csv.writer(f).writerow(row)

    def log_full_game(self, history, log_str):
        print(f"Logging full game for {log_str}")
        def numpy_handler(obj):
            if hasattr(obj, 'item'):
                return obj.item()
            raise TypeError(f"Not serializable: {obj.__class__.__name__}")
        with open(self.move_log_path, "a", buffering=1) as f:
            for entry in history:
                f.write(json.dumps(entry, default=numpy_handler) + "\n")

    # ── Action Selection ───────────────────────────────────────────────────────
    def get_masked_q_values(self, state, mask):
        """
        Single forward pass through policy_net.
        Returns (masked_q, raw_q) so callers never need a second forward pass.

        """
        mask_t = torch.from_numpy(mask).to(self.device)
        # Pass the board through the policy net to assess Q(s,a) for the 4 actions
        with torch.no_grad(): # This is an inference call and so no graident updates
            q_values = self.policy_net(state.to(self.device))
        masked_q = q_values.clone()
        masked_q[0, ~mask_t] = -float('inf')
        return masked_q, q_values

    # Choose the action with the largest Q(s,a) with probability (1-epsilon) and a completely random move with probability epsilon
    def select_action(self, masked_q_values, epsilon, mask):
        """Epsilon-greedy selection using pre-computed masked Q-values."""
        if random.random() > epsilon:
            return masked_q_values.max(1)[1].view(1, 1), True
        valid_indices = np.where(mask)[0]
        action = random.choice(valid_indices) if len(valid_indices) > 0 else 0
        return torch.tensor([[action]], device=self.device, dtype=torch.long), False

    # ── Optimisation 
    # This will sample from the buffer, do one set of weight updates based on the samples, do one soft transfer from policy to target
    def optimize_model(self):
        if len(self.memory) < BATCH_SIZE: # Wait for the buffer to fill up
            return None

        batch = self.memory.sample(BATCH_SIZE)
        sb  = batch['states']
        ab  = batch['actions']
        rb  = batch['rewards']
        nsb = batch['next_states']
        db  = batch['dones']
        nmb = batch['next_masks']

        # Pass all boards in the mini-batch to the policy net to see what it predicts 
        # Note that this is not wrapped within no_grad() because this is the model prediction that will be compared to the label
        # And the loss will drive the gradient descent and update the policy net
        current_q_values = self.policy_net(sb).gather(1, ab) # the gather() part is a vectorized part of indexing each row by a different column

        # Note that the target net does not get updated by gradient descent and so we wrap this within no_grad()
        with torch.no_grad():
            next_q_all = self.target_net(nsb) # Evaluate the next state through the target net
            next_q_all[~nmb] = -float('inf')
            next_q_values = next_q_all.max(1)[0] # The best action as per the target net
            next_q_values = torch.where(
                db.bool(), torch.zeros_like(next_q_values), next_q_values
            )

        expected_q_values = rb + GAMMA * next_q_values # This is the label or the dependent variable
        # Huber loss because here, even labels are very noisy and we don't want to heavily penalize the model for errors (especially early-on)
        loss = F.smooth_l1_loss(current_q_values, expected_q_values.unsqueeze(1)) 

        self.optimizer.zero_grad()
        loss.backward() # calculate loss gradients
        self.optimizer.step() # apply weight updates based on calculated gradients and the Adam optimizer

        # Soft update of target network
        with torch.no_grad():
            for t_p, p_p in zip(self.target_net.parameters(),
                                 self.policy_net.parameters()):
                t_p.data.copy_(self.tau * p_p.data + (1.0 - self.tau) * t_p.data)

        return loss.item()

    # ── Reward Shaping ─────────────────────────────────────────────────────────
    # Ideally, the reward from the game engine should be enough for the model to train
    # But here, we put our thumb on the scale and nudge the model towards better strategies
    def get_shaped_reward(self, board, base_reward, moved):
        if not moved:
            self.log_game_flag = True
            print("WARNING: moved=False despite action masking — check env.")
            return -1.0

        shaped = 0.0
        # Too much numerical range in the base reward that can lead to training instability. Take log2 for better numerical properties
        log_tile_reward = 0.0
        if base_reward > 0:
            log_tile_reward = np.log2(base_reward) 
            shaped += log_tile_reward

        # Encourage the agent to create empty cells which give room for maneuverability
        empty_cells = np.count_nonzero(board == 0)
        empty_cell_reward = empty_cells * 0.1 # weight of 0.1 here is empirical
        shaped += empty_cell_reward

        # Encourage the agent to park the highest tiles in the corner
        # For any board, compute a weighted score based on this table, compare it to the largest achievable score
        # which will be when the tiles of this board are ordered in descending order from the top left
        # and use this fraction as incentive
        weights = np.array([
                    [7,   6,   5,   4  ],
                    [ 3,   3,   3,   3  ],
                    [ 1, 1, 1, 1],
                    [ 0,   0,   0,   0  ],
                ])
        tiles   = np.sort(board.flatten())[::-1]           # tiles largest first
        wts     = np.sort(weights.flatten())[::-1]         # weights largest first
        max_possible = np.sum(tiles * wts)                 # optimal assignment
        tile_layout_reward = 0.0
        if max_possible > 0:
            tile_layout_reward = (np.sum(board * weights) / max_possible) * 5.0 # Again, the weight of 5.0 here is empirical
            shaped += tile_layout_reward

        # Incentivize model to line up identical tiles horizontally or vertically
        h_pairs = np.sum((board[:, :3] == board[:, 1:]) & (board[:, :3] != 0))  # horizontal
        v_pairs = np.sum((board[:3, :] == board[1:, :]) & (board[:3, :] != 0))  # vertical
        alignment_reward = (h_pairs + v_pairs) * 1.0
        shaped += alignment_reward

        # Log the breakdown to manually inspect and adjust weights as needed
        shaped_reward_components = {
            "log_tile_reward"       : log_tile_reward,
            "empty_cell_reward"     : empty_cell_reward,
            "tile_layout_reward"    : tile_layout_reward,
            "alignment_reward"      : alignment_reward
        }
        
        return shaped, shaped_reward_components

    # ── Training Loop ──────────────────────────────────────────────────────────
    def train(self, num_episodes=10):
        start_time   = time.perf_counter()
        env          = Game2048()
        action_names = {0: "LEFT", 1: "UP", 2: "RIGHT", 3: "DOWN"}
        best_tile    = 0

        for episode in range(self.start_episode, self.start_episode + num_episodes):
            board = env.reset()
            done  = False
            self.log_game_flag = False

            episode_history     = []
            episode_losses      = []
            episode_q_maxes     = []
            total_reward        = 0.0
            total_shaped_reward = 0.0

            while not done:
                # Determine the level of exploration
                epsilon = EPS_END + (EPS_START - EPS_END) * np.exp(-self.steps_done / EPS_DECAY)
                
                state = preprocess_cnn(board)
                mask  = env.get_valid_moves()

                masked_q, raw_q     = self.get_masked_q_values(state, mask) # get Q(s,a) for all valid a
                action, is_greedy   = self.select_action(masked_q, epsilon, mask) # move in an epsilon-greedy fashion based on Q(s,a)

                episode_q_maxes.append(masked_q.max().item()) # logging

                # Game engine will move as per chosen action, advance the board, and spawn a new tile at random
                next_board, base_reward, done, moved, next_mask, new_tile_loc = env.move(action.item())
                shaped_reward, shaped_reward_components = self.get_shaped_reward(next_board, base_reward, moved)
                next_state = preprocess_cnn(next_board)

                # Store this move in the memory buffer for future training
                self.memory.push(
                    state, action, next_state,
                    shaped_reward, float(done), next_mask
                )

                masked_q_for_logging = [None if math.isinf(x) else x for x in masked_q.flatten().tolist()] # R script doesn't like "-Infinity"
                episode_data = {
                    "episode"       : episode,
                    "epsilon"       : epsilon,
                    "MoveNum"       : self.steps_done,
                    "board"         : board.flatten().tolist(),
                    "action"        : action.item(),
                    "is_greedy"     : is_greedy,
                    "base_reward"   : base_reward,
                    "shaped_reward" : shaped_reward,
                    "masked_q"      : masked_q_for_logging,
                    "new_tile_loc"  : new_tile_loc
                }
                episode_data.update(shaped_reward_components)
                episode_history.append(episode_data)

                if self.verbose: # Allows us to inspect every move
                    print(board)
                    print(f"Action: {action_names[action.item()]} | "
                          f"Moved: {moved} | New Tile Location: {new_tile_loc}")
                    print(episode_data)
                    print(next_board)
                    print("--------------------")
                    #input("Press Enter to continue...")

                board               = next_board
                total_reward        += base_reward
                total_shaped_reward += shaped_reward
                self.steps_done     += 1
                self.steps_this_run += 1

                # Update both models (policy net through SGD and target net through polyak transfer) every so often
                if self.steps_done % OPTIMIZE_EVERY == 0:
                    loss = self.optimize_model()
                    if loss is not None:
                        episode_losses.append(loss)

                if done:
                    # Game over. Take care of logging, model checkpointing etc.
                    final_max = board.max()
                    self.max_tile_history.append(final_max)
                    self.distribution[int(final_max)] += 1
                    avg_loss = float(np.mean(episode_losses)) if episode_losses else 0.0
                    avg_q    = float(np.mean(episode_q_maxes)) if episode_q_maxes else 0.0

                    self.log_episode_stats(
                        episode, total_reward, total_shaped_reward,
                        final_max, avg_loss, avg_q, self.steps_done, board
                    )

                    # Need to log the very last board which now lives within board because we have already assigned board = next_board
                    masked_q_for_logging = [None if math.isinf(x) else x for x in masked_q.flatten().tolist()]
                    episode_data = {
                        "episode"       : episode,
                        "epsilon"       : epsilon,
                        "MoveNum"       : self.steps_done,
                        "board"         : board.flatten().tolist(),
                        "action"        : action.item(),
                        "is_greedy"     : is_greedy,
                        "base_reward"   : base_reward,
                        "shaped_reward" : shaped_reward,
                        "masked_q"      : masked_q_for_logging,
                        "new_tile_loc"  : new_tile_loc
                    }
                    episode_data.update(shaped_reward_components)
                    episode_history.append(episode_data)

                    # Log full game either probabilistically (so that we get an idea of how games end in different ways)
                    # or log all games that exceed a certain tile
                    log_str = f"max tile {final_max}"
                    if self.probs_for_logging is None:
                        if final_max >= self.min_log_tile:
                            self.log_full_game(episode_history, log_str)
                    else:
                        prob = float(self.probs_for_logging.get(int(final_max), 1.0)) # If no prob found, always log the game with probability 1
                        if random.random() < prob:
                            log_str = f"{log_str} with logging probability {prob}"
                            self.log_full_game(episode_history, log_str)

                    is_best = final_max > best_tile
                    if is_best:
                        best_tile = final_max
                        self.save_checkpoint(episode, final_max, is_best=True)
                    elif episode % 500 == 499:
                        print(f"Saving periodic checkpoint at episode {episode + 1}")
                        self.save_checkpoint(episode, final_max, is_best=False)

                    if episode % 500 == 499:
                        elapsed = time.perf_counter() - start_time
                        print(
                            f"Ep {episode+1:>6} | Steps {self.steps_done:>8} | "
                            f"Score {total_reward:>7.0f} | Tile {final_max:>4} | "
                            f"e {epsilon:.3f} | Loss {avg_loss:.5f} | "
                            f"Q {avg_q:>8.2f} | {elapsed:.1f}s"
                        )
                        for tile in sorted(self.distribution):
                            cnt = self.distribution[tile]
                            pct = cnt / len(self.max_tile_history) * 100
                            print(f"  Tile {tile:4d}: {cnt:5d} games ({pct:.1f}%)")
                    break


### 6. Run Training

Call the training code. We can run this in batches checkpointing as we go along. Only challenge with this is that the memory buffer starts empty with each run. Though it needs only 64 moves before training can start, we ideally want iid samples for optimal training (and consecutive moves of a game won't be iid) which means the first few games are not being optimally used every time we train from a checkpoint. But this effect is negligible if we are training for thousands of games.

In [ ]:
if __name__ == "__main__" or True:
    RUN_DESC   = 'fresh_allshapes_lindecay' # All components of reward shaping + eps decaying exponentially (no cosine scheduling etc.)
    LOG_FOLDER = Path('results/run_04aj/')
    LOG_FOLDER.mkdir(parents=True, exist_ok=True)

    trainer = Trainer(
        min_log_tile        = 2048, # superfluous if probabilistic logging is enabled
        probs_for_logging   = {2: 1.0, 4: 1.0, 8: 1.0, 16: 1.0, 32: 0.02, 64: 0.01, 128: 0.004, 256: 0.002, 512: 0.002, 1024: 0.001, 2048: 0.005, 4096: 1},
        resume_path         = 'results/run_04ai/latest_model_fresh_allshapes_lindecay.pth',
        #resume_path         = None,
        run_folder          = str(LOG_FOLDER),
        stats_log_file_name = f"Stats_{RUN_DESC}.csv",
        moves_log_file_name = f"Moves_{RUN_DESC}.jsonl",
        chkpoint_suffix     = RUN_DESC,
        verbose             = False,
    )
    trainer.train(num_episodes=12_000)

## Inference

There is a crucial difference between the games played by the agent during training and inference. For the purposes of inference, $\epsilon$ can be set to $0$. In other words, we always follow the model's recommendations and we do not do any greedy explorations. In the training above, we capped $\epsilon$ from below at $0.01$ but even this low value can have an impact. A game that forms the 1024 tile typically lasts $600 - 800$ moves and across this span, we will do $6-8$ random moves at $\epsilon = 0.01$. While this might not seem like a lot, a bad move at a late stage on a crowded board with big tiles can be a game killer. You will see examples of this below.

In [ ]:
import pandas as pd

def preprocess_board(board: np.ndarray) -> torch.Tensor:
    """Log2 scaling and normalization to [0, 1] for 4x4 board input."""
    log_board = np.where(board > 0, np.log2(board), 0.0)
    norm_board = log_board / 11.0 
    
    # Reshape to 4D Tensor: (Batch_Size=1, Channels=1, Height=4, Width=4)
    return torch.tensor(norm_board, dtype=torch.float32).unsqueeze(0).unsqueeze(0)

def run_inference(
    model_class, 
    checkpoint_path: str, 
    initial_board: np.ndarray = None, 
    num_games: int = 10,
    epsilon: float = 0.0,
    move_step_by_step: bool = False,
    print_every_game: bool = True,
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
):
    if move_step_by_step & (num_games != 1):
        print("EXITING: step_by_step cannot be enabled for multi-game scenarios")
        return -1.0
    # 1. Instantiate and load model in evaluation mode
    model = model_class().to(device)
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    
    if isinstance(checkpoint, dict) and 'state_dict' in checkpoint:
        model.load_state_dict(checkpoint['state_dict'])
    else:
        model.load_state_dict(checkpoint)
        
    model.eval()

    results = []

    # 2. Execute N games
    for game_idx in range(num_games):
        env = Game2048()
        
        # Override starting board only if a custom board layout is provided
        if initial_board is not None:
            env.board = np.copy(initial_board)
        
        done = False
        total_score = 0
        moves = 0

        while not done:
            state_tensor = preprocess_board(env.board).to(device)
            valid_mask = env.get_valid_moves()  # Boolean array of shape (4,)

            if not any(valid_mask):
                break

            # Forward pass (Deterministic Greedy Action Selection)
            with torch.no_grad():
                q_values = model(state_tensor).squeeze(0)  # Shape: (4,)

            # Mask invalid actions
            q_values[~torch.tensor(valid_mask, device=device)] = -float('inf')

            if epsilon > 0 and random.random() < epsilon:
                # Random move, uniform over VALID actions only
                valid_indices = np.where(valid_mask)[0]
                action = torch.tensor(random.choice(valid_indices), device=device)
            else:
                # Greedy: highest predicted Q-value
                action = torch.argmax(q_values)
            
            # Execute move in environment
            next_board, base_reward, done, moved, next_mask, new_tile_loc = env.move(action.item())
            total_score += base_reward
            moves += 1

            if move_step_by_step:
                print(f"Move = {action}. Reward = {base_reward}. New Tile location = {new_tile_loc}. Done = {done}")
                show_board(next_board)
                input("Press any key to continue")

        max_tile = int(np.max(env.board))
        results.append({"game": game_idx + 1, "score": total_score, "target_value": max_tile, "steps": moves})
        if print_every_game:
            print(f"Game {game_idx + 1:2d} | Max Tile: {max_tile:4d} | Score: {total_score:6d} | Moves: {moves:4d}")

    return pd.DataFrame(results)

In [ ]:
# Play games starting from the standard default board (initial_board=None)
inf_results = run_inference(DQN_CNN, "results/run_04aj/latest_model_fresh_allshapes_lindecay.pth", num_games=5000, epsilon = 0.0, print_every_game = False)
print(inf_results['target_value'].value_counts().sort_index())

# Play games starting from a specific starting point with more output logging
custom_start = np.array([[2,16,4,8], [4,2,32,2], [8,16,2,16], [2,2,8,4]], dtype=np.int32)
run_inference(DQN_CNN, "results/run_04aj/latest_model_fresh_allshapes_lindecay.pth", initial_board=custom_start, num_games=1, move_step_by_step = True)

### Systematic inference runs across selected models

Code to run multiple inference runs at each checkpoint

In [ ]:
import re
np.seterr(divide='ignore')

folder_list = ['aa','ab','ac','ad','ae','af','ag','ah','ai','aj']
NUM_INF_RUNS = 1000
folders = [f'results/run_04{f}/' for f in folder_list]
run_count = 1
for curr_folder in folders:
    all_files_this_folder = os.listdir(curr_folder)
    models_this_folder = [f for f in all_files_this_folder if re.search('periodic_', f)]

    for curr_model in models_this_folder:
        print(f"Run count = {run_count}. Processing inferene runs for model {curr_model} in folder {curr_folder}")
        run_count = run_count + 1
        if True: # Set to False just to see how many models there are within these folders
            this_model_inf = run_inference(
                DQN_CNN, curr_folder + curr_model,
                print_every_game=False, num_games=NUM_INF_RUNS, epsilon=0.00
            )
            chkpt_num = int(re.sub(r".*_ep(\d+)\.pth$", r"\1", curr_model))
            this_model_inf['chkpt'] = chkpt_num
            this_model_inf.to_csv(
                f'{curr_folder}Infruns_{NUM_INF_RUNS}_chkpt_{chkpt_num}.csv',
                index=False
            )

## Results

Both training and inference can be time consuming processes. The training throughput is roughly $1500$ games/hour and the inference throughput is around $6500$ games/hour. As the model gets better, the games get longer, and these numbers might come down a little. In this section, we present results generated using an agent trained for $500,000$ games and with $1000$ inference runs generated using checkpoints created once every $5000$ games.

### Overall Performance

<img src="figures/tile_race.gif" style="width:700px; max-width:100%">

*Figure 1: Proportion of tiles in the most recent 5,000 episodes of training*

Like we discussed earlier, training includes $\epsilon$-greedy exploration which can destroy promising boards with a random, sub-optimal move. During inference, we set $\epsilon = 0.0$ and we can see that this helps the agent achieve significantly higher proportions of high value tiles. Games that produce high value tiles tend to last many moves (a game that reaches $2048$ lasts about 1300 moves on average) which means the $1\%$ random $\epsilon$-exploration has $10+$ opportunities to wreak havoc.

<img src="figures/Proportions_over_time_animation.gif" style="width:700px; max-width:100%">

*Figure 2: Performance Comparison between training and inference*

As an aside, we can get a baseline performance of how a fully random game is likely to unfold simply by running the inference engine with $\epsilon$ set to $1.0$. A random agent achieves the following distribution:

| Max Tile | Proportion of games
| :------: | :--------:
| 16     | < 0.5%
| 32     |   7%  
| 64     |  38%  
| 128    |  46% 
| 256    |  8% 
| 512    | < 0.1%

We can see from the curves above that the agent betters this random baseline pretty much right from the start (as we would expect)

### Analysis of game logs

The code logs detailed move-level statistics on a bunch of metrics - it logs the state of the board $s$ at each move (values in all $16$ squares), $Q(s,a)$ for each of the 4 actions, value of $\epsilon$, whether the move is random (which happens with probability $\epsilon$), the overall reward for the move and the different constituents of reward shaping. We can gain insights into the workings of the algorithm by analyzing these logs.

We are going to look at two different games in this analysis - game $59885$ early on where the model got till the 512 tile and game $438754$ where the model got to the 4096 tile.

#### 1. Evolution of $Q(s,a)$ over moves

This plot shows how $Q(s,a)$ evolved with every move for these two games.

<img src="figures/Qvalues_vs_moves.png" style="width:1200px; max-width:100%">

*Figure 3: Evolution of $Q(s,a)$ as a game progresses*

Couple of things pop out immediately from the plot.
* Obviously, the 512 tile game lasts for much less time than the 4096 tile game
* There are some gaps in the data for some of the curves - these are game states where the corresponding moves would have been invalid (which are logged as NA)
* Perhaps surprisingly, $Q(s,a)$ is practically indistinguishable for $3$ of the $4$ moves in both games. It is clear that DOWN is not preferred at all and this is very likely due to the tile layout reward component. The last row of that weight matrix is $0$ which strongly discourages the model from moving any high value tile downwards.
* If $3$ out of the $4$ moves are indistinguishable, how is the model working at all? The figure below shows this in even starker terms

<img src="figures/Qvalues_PercGapTop2.png" style="width:900px; max-width:100%">

*Figure 4: Percentage gap between the top two best actions*

This plot was created as follows: For each move, compute the percentage difference between the largest and the second largest values of $Q(s,a)$ (which is a proxy for the model's discriminatory power). To see if this discriminatory power changes as the game evolves, we use the largest tile on the board as a proxy for game stage. We take a median of this percentage difference for all moves where the largest tile on board was some $2^k$.

* The difference between the best and the second best move is minimal, around $0.2\%$
* There seems to be no great difference across game stages. In longer games, there is perhaps a wee bit more discriminatory power at the beginning and end stages (where the best move separates itself a bit more from the second best move) but not by much.

This small delta is because of a couple of reasons, some structural and others due to sub-optimal design choices. 

* For most of the game, a LEFT vs. a RIGHT is not ultra-critical. One might be slightly more optimal than the other but it is not like chess where one wrong move is a killer. Accordingly, we should expect small differences in the numerator of this percentage change metric. We can argue that the agent just needs _some_ difference to select an optimal move and not necessarily a _large_ difference.
* The denominator (total reward $Q(s,a)$) is very large since it calculates reward over hundreds of moves. The mean reward per move is around $12-16$ for most games which makes the denominator much larger than the numerator.
* There is also the sub-optimal reward shaping (see next section) which gives a disproportionate amount of reward for the layout of the board. As long as the board layout is maintained (highest tile in the north-west corner), $Q(s,a)$ and $Q(s',a')$ will be very high and very close regardless of what action the agent takes. This is not a fatal flaw in the sense that the network continues to improve with more training but there is definitely room for optimization here.

#### 2. Components of Reward Shaping

Recall that the reward given to the agent for any move contains $4$ components:
* $\log2(\cdot)$ of the game engine's reward for that move (which is the sum of all the new tiles formed by that move)
* An incentive for having empty tiles (which gives the model more room for maneuverability)
* An incentive for aligning identical tiles horizontally or vertically (which gives the model valid moves and creates larger tiles in the process)
* An incentive to keep the largest tiles in the top left of the board. This is the kind of strategy I would have ideally liked the model to learn by itself.

<img src="figures/reward_shaping_components.png" style="width:900px; max-width:100%">

*Figure 5: Components of the agent's reward*

The plot shows the relative contribution of these $4$ components to the total cumulative reward (the instantaneous reward are very spiky and noisy which doesn't make for a readable plot). Direct reward from the game engine is only around $25\%$ of the total and tile layout is nearly $50\%$ of the rewards, which feels too high. Ultimately though, there is no "right" distribution here and the relative weights for these reward components is an empirical question.

#### 3. True vs. Predicted Rewards

The policy net is trained to predict $Q(s,a) = r + \gamma \times \max_{a'}Q(s',a')$ and the logs give us an opportunity to check this fit. For any move $i$ with board state $s$, we are aleady logging $Q(s,\cdot)$ and the shaped reward $r$. The term $\max_{a'}Q(s',a')$ is the reward estimate from the next move onwards which is just $Q(s,a)$ from row $(i+1)$ (with the caveat that if we choose an exploratory step here with probability $\epsilon$, this is no longer true. But we ignore this because it would only affect around $1\%$ of the points)

But unlike typical machine learning problems, the comparison here is not so straightforward.
* Remember that $Q(s,a)$ is an expectation over future random tile spawns but here we have a single realization.
* We need to evaluate the fit carefully because typically $r$ is very small (a typical move merges small tiles and even the $90^{\text{th}}$ percentile of rewards is only $36$) whereas $Q(s,a)$ and $Q(s',a')$ are much larger (in the order of several hundreds to $1000+$). A naive comparison between the two will look very good simply because of this magnitude difference.

<img src="figures/per_move_true_vs_predicted.png" style="width:1200px; max-width:100%">

*Figure 6: Max $Q(s,a)$ for this move vs. Predicted $(r + \gamma \max_{a'} Q(s',a'))$*

This comparison looks great but is misleading. The small deviations from the diagonal are where the story lies and to see that better, we can compare the residual between the true label and predicted label and look at its magnitude wrt the immediate reward $r$. The immediate reward from the game engine is the one true signal in the agent's learning process and if the residual is comparable to it, the noise in the system is drowning out the signal.

<img src="figures/ratio_of_residuals.png" style="width:1200px; max-width:100%">

*Figure 7: Proportion of residual wrt immediate reward*

This tells a clearer story. Most of the moves lie within the $[-1,1]$ band (which means the residual noise is not overwhelming the reward signal) but it is still a very noisy environment. We can also see that as the game gets tougher and nears an end, the errors pile up (many dots corresponding to 2048 and 4096 tiles lie outside the $[-1,1]$ band)

## Learnings and Next Steps

My biggest learning is to develop an appreciation for how subtle and difficult reinforcement learning is. I had enough online resources (and LLMs) to consult at each stage and even then, this often felt like an impossible task. I trained this model for over $15$ days of CPU time and it was only around the $10^{\text{th}}$ day that I could see the 2048 tile emerge consistently. Also, the performance is very sensitive to hyperparameter choices ($\epsilon$-decay schedule, buffer sizes etc.) - I _think_ the algorithm itself is robust enough and will eventually achieve the 2048 tile, but to do it in the shortest time requires both deep understanding and lots of experimentation.

I thought this was a great problem to learn RL on. For many of the reasons listed above, it is genuinely difficult and forces you to engage with the main ideas of the field. At the same time, it is also easy enough that it can be solved on a laptop (albeit with nearly a month of training) and it neatly motivates the need for more advanced forms of RL to tackle harder problems.

Time permitting, I would love to explore the following:
* **Ablation studies** - run this on the cloud and experiment with different hyperparameter settings. I am especially curious about the $\epsilon$-schedule and with replacing the CNN with larger models.
* **Preferential buffer recall** - With some probability, we can show that all moves logged in the buffer have the same expected number of times that they are used in training. As an interesting aside, this number is independent of buffer size and works out to $(B/K)$ where $B$ is the minibatch size and we optimize weights once every $K$ moves. Remember that this buffer has _all_ moves and board states including trivial ones at the beginning of a game. A sophisticated agent that can consistently get to the 512 tile might be better served by training more on complex late-stage moves than early-stage moves. Operationally, this can be done by non-uniform sampling based on the largest tile in the board. It will be interesting to see if this helps speed up training. Another similar idea is to skip pushing the early moves into the buffer once the agent can reliably go past them.
* **CNNs under the hood** - Can we visualize the internal representations of the CNN to figure out what in the board it is responding to?
* **N-step Q Learning** - Instead of learning just the immediate reward from the game engine, play the game for $N$ moves and collate the rewards. Use the target net to predict $Q(s_{t+N},\cdot)$ with a discount factor of $\gamma^N$. This makes learning slower (need to play multiple moves to get a single label) but more robust (the noisier future predictions get discounted more).
* **Other techniques** - We have used a relatively basic version of RL. Can we use this to learn more sophisticated versions of RL (like PPO)? Can we "learn" the optimal $\epsilon$-schedule alongside the policy as opposed to imposing it from the outside? Also, not everything has to be about deep learning. There are apparently very strong model-free techniques (Expectimax algorithms) that are very good at 2048.
* **Code vectorization** - The code as it stands now is highly sequential with training and game play interleaved sequentially. It _should_ be possible to vectorize large parts of this by playing multiple games in parallel, filling up the buffer faster, training the model more efficiently through GPUs etc. I went down this path but couldn't get the training to work well and eventually gave up in favour of a simpler architecture. But even a 2-3x speedup opens up significantly more experimentation possibilities compared to the current architecture.